In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
sys.path.append(os.path.abspath(".."))
os.chdir(os.path.abspath(".."))

with open("keys/openai_api_key.txt") as f:
    os.environ["OPENAI_API_KEY"] = f.read().strip()

with open("keys/hf_token.txt") as f:
    os.environ["HF_TOKEN"] = f.read().strip()
    os.environ["HF_HOME"] = "/network/scratch/t/tejas.kasetty/huggingface"

#### Data and Prompt Generation

In [2]:
from src.dataset import Dataset, ShiftCipher, Datasets
from src.generate import ShiftCipherGenerator, generate_prompts

sc = ShiftCipher(100, 1000, word_length=1)
data = list(sc.sample(1))
# sc_gen = ShiftCipherGenerator()
# sc_gen.generate(data)
dataset_type = Datasets.SHIFT_CIPHER.value
system, prompts = generate_prompts(dataset_type, data)

In [3]:
system, prompts

("Task: Shift Cipher Prediction\n\nYou are given a set of input-output examples where each input word X is transformed into an output word Y by applying a simple letter shift cipher.\nIn each example, every letter in X has been shifted by a fixed number of steps forward in the alphabet to produce Y (wrapping around if needed, e.g., 'z' becomes 'a').\n\nThe demonstrations show correctly paired examples:\nX: abc\nY: xyz\n\nX: def\nY:\n\nYour goal -\nBased on the pattern observed from the demonstrations, predict the correct Y for a final unseen input X.\n-----------------------------\n",
 [          context    query
  0    X:V\nY:Q\n\n  X:R\nY:
  1    X:R\nY:M\n\n  X:I\nY:
  2    X:I\nY:D\n\n  X:G\nY:
  3    X:G\nY:B\n\n  X:N\nY:
  4    X:N\nY:I\n\n  X:P\nY:
  ..            ...      ...
  994  X:K\nY:F\n\n  X:R\nY:
  995  X:R\nY:M\n\n  X:J\nY:
  996  X:J\nY:E\n\n  X:U\nY:
  997  X:U\nY:P\n\n  X:L\nY:
  998  X:L\nY:G\n\n  X:E\nY:
  
  [999 rows x 2 columns],
            context    query
  

### Initialize model

In [4]:
llm_names = ["meta-llama/Llama-2-7b-chat-hf", "meta-llama/Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-v0.1", "mistralai/Mistral-7B-Instruct-v0.1", "mistralai/Mistral-7B-Instruct-v0.2", "mistralai/Mistral-7B-v0.3", "mistralai/Ministral-8B-Instruct-2410"]

In [4]:
from src.llms import Model, get_model

llm_names = Model.list()
model_name = llm_names[5]
model = get_model(model_name)


/home/mila/t/tejas.kasetty/.conda/envs/preq_code/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 04-30 13:11:28 [__init__.py:239] Automatically detected platform cuda.


2025-04-30 13:11:29,660	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 04-30 13:11:38 [config.py:689] This model supports multiple tasks: {'generate', 'embed', 'score', 'reward', 'classify'}. Defaulting to 'generate'.
INFO 04-30 13:11:38 [config.py:1901] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-30 13:11:39 [core.py:61] Initializing a V1 LLM engine (v0.8.4) with config: model='meta-llama/Meta-Llama-3.1-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Meta-Llama-3.1-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='auto', reasoning_backend=None), observability_config=ObservabilityConfig(show_hidden_metri

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:00<00:00,  9.43it/s]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:00<00:00,  3.27it/s]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:01<00:00,  2.29it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:01<00:00,  2.02it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:01<00:00,  2.31it/s]



INFO 04-30 13:11:45 [loader.py:458] Loading weights took 1.92 seconds
INFO 04-30 13:11:46 [gpu_model_runner.py:1291] Model loading took 14.9889 GiB and 2.671601 seconds
INFO 04-30 13:11:53 [backends.py:416] Using cache directory: /home/mila/t/tejas.kasetty/.cache/vllm/torch_compile_cache/3a982973f7/rank_0_0 for vLLM's torch.compile
INFO 04-30 13:11:53 [backends.py:426] Dynamo bytecode transform time: 7.67 s
INFO 04-30 13:11:54 [backends.py:115] Directly load the compiled graph for shape None from the cache
INFO 04-30 13:11:59 [monitor.py:33] torch.compile takes 7.67 s in total
INFO 04-30 13:12:01 [kv_cache_utils.py:634] GPU KV cache size: 154,432 tokens
INFO 04-30 13:12:01 [kv_cache_utils.py:637] Maximum concurrency for 131,072 tokens per request: 1.18x
INFO 04-30 13:12:50 [gpu_model_runner.py:1626] Graph capturing finished in 49 secs, took 1.61 GiB
INFO 04-30 13:12:50 [core.py:163] init engine (profile, create kv cache, warmup model) took 64.88 seconds
INFO 04-30 13:12:50 [core_client

### Query and Get Results

In [5]:
from src.query import Query
results, query_stats = Query(model, dataset_type, 1).query_prompts(system, prompts)

Processed prompts: 100%|██████████| 999/999 [00:00<00:00, 15289.25it/s, est. speed input: 54402596.00 toks/s, output: 15365.78 toks/s]


Error - query 2 failed. Could not parse response after 0 retries.
Error - query 143 failed. Could not parse response after 0 retries.
Error - query 240 failed. Could not parse response after 0 retries.
Error - query 247 failed. Could not parse response after 0 retries.
Error - query 248 failed. Could not parse response after 0 retries.
Error - query 590 failed. Could not parse response after 0 retries.
Error - query 597 failed. Could not parse response after 0 retries.
Error - query 844 failed. Could not parse response after 0 retries.
Error - query 848 failed. Could not parse response after 0 retries.
Error - query 945 failed. Could not parse response after 0 retries.
Error - query 959 failed. Could not parse response after 0 retries.


Processed prompts: 100%|██████████| 999/999 [00:00<00:00, 13724.43it/s, est. speed input: 44493403.10 toks/s, output: 13741.58 toks/s]


Error - query 34 failed. Could not parse response after 0 retries.
Error - query 35 failed. Could not parse response after 0 retries.
Error - query 36 failed. Could not parse response after 0 retries.
Error - query 48 failed. Could not parse response after 0 retries.
Error - query 49 failed. Could not parse response after 0 retries.
Error - query 59 failed. Could not parse response after 0 retries.
Error - query 62 failed. Could not parse response after 0 retries.
Error - query 67 failed. Could not parse response after 0 retries.
Error - query 72 failed. Could not parse response after 0 retries.
Error - query 74 failed. Could not parse response after 0 retries.
Error - query 85 failed. Could not parse response after 0 retries.
Error - query 92 failed. Could not parse response after 0 retries.
Error - query 105 failed. Could not parse response after 0 retries.
Error - query 108 failed. Could not parse response after 0 retries.
Error - query 115 failed. Could not parse response after 0 r

Processed prompts: 100%|██████████| 999/999 [00:00<00:00, 17777.22it/s, est. speed input: 63353346.59 toks/s, output: 17812.67 toks/s]


...

Error - query 33 failed. Could not parse response after 0 retries.
Error - query 36 failed. Could not parse response after 0 retries.
Error - query 39 failed. Could not parse response after 0 retries.
Error - query 42 failed. Could not parse response after 0 retries.
Error - query 43 failed. Could not parse response after 0 retries.
Error - query 46 failed. Could not parse response after 0 retries.
Error - query 47 failed. Could not parse response after 0 retries.
Error - query 49 failed. Could not parse response after 0 retries.
Error - query 57 failed. Could not parse response after 0 retries.
Error - query 60 failed. Could not parse response after 0 retries.
Error - query 80 failed. Could not parse response after 0 retries.
Error - query 100 failed. Could not parse response after 0 retries.
Error - query 103 failed. Could not parse response after 0 retries.
Error - query 104 failed. Could not parse response after 0 retries.
Error - query 111 failed. Could not parse response after 0 

Processed prompts: 100%|██████████| 999/999 [00:00<00:00, 15096.34it/s, est. speed input: 49086672.20 toks/s, output: 15158.98 toks/s]


Error - query 0 failed. Could not parse response after 0 retries.
Error - query 56 failed. Could not parse response after 0 retries.
Error - query 57 failed. Could not parse response after 0 retries.
Error - query 63 failed. Could not parse response after 0 retries.
Error - query 69 failed. Could not parse response after 0 retries.
Error - query 70 failed. Could not parse response after 0 retries.
Error - query 89 failed. Could not parse response after 0 retries.
Error - query 97 failed. Could not parse response after 0 retries.
Error - query 99 failed. Could not parse response after 0 retries.
Error - query 101 failed. Could not parse response after 0 retries.
Error - query 102 failed. Could not parse response after 0 retries.
Error - query 109 failed. Could not parse response after 0 retries.
Error - query 114 failed. Could not parse response after 0 retries.
Error - query 144 failed. Could not parse response after 0 retries.
Error - query 148 failed. Could not parse response after 0

Processed prompts: 100%|██████████| 999/999 [00:00<00:00, 1550.50it/s, est. speed input: 5511909.66 toks/s, output: 1550.65 toks/s]

Error - query 440 failed. Could not parse response after 0 retries.
Error - query 628 failed. Could not parse response after 0 retries.
Error - query 630 failed. Could not parse response after 0 retries.
Error - query 640 failed. Could not parse response after 0 retries.
Error - query 655 failed. Could not parse response after 0 retries.
Error - query 729 failed. Could not parse response after 0 retries.
Error - query 730 failed. Could not parse response after 0 retries.
Error - query 731 failed. Could not parse response after 0 retries.
Error - query 845 failed. Could not parse response after 0 retries.


In [32]:
results, query_stats

([[{'answer': ['u'],
    'logprobs': [[('u', -1.0691),
      ('q', -1.1941),
      ('?ĊĊ', -2.5066),
      ('z', -2.6941),
      ('Ġ?ĊĊ', -3.4441)]],
    'is_valid': True},
   {'answer': ['u'],
    'logprobs': [[('u', -1.405),
      ('?ĊĊ', -1.4675),
      ('q', -2.03),
      ('Ġ?ĊĊ', -2.3425),
      ('?Ċ', -3.53)]],
    'is_valid': True},
   {'answer': ['u'],
    'logprobs': [[('u', -0.788),
      ('q', -1.663),
      ('z', -1.788),
      ('?ĊĊ', -3.1005),
      ('Ġ?ĊĊ', -3.538)]],
    'is_valid': True},
   {'answer': ['z'],
    'logprobs': [[('z', -0.7733),
      ('q', -1.7733),
      ('u', -2.1483),
      ('?ĊĊ', -2.4608),
      ('?Ċ', -3.0233)]],
    'is_valid': True},
   {'answer': ['q'],
    'logprobs': [[('q', -0.5701),
      ('u', -1.6326),
      ('?ĊĊ', -2.8826),
      ('z', -4.0701),
      ('Ġ?ĊĊ', -4.1951)]],
    'is_valid': True},
   {'answer': ['q'],
    'logprobs': [[('q', -0.6662),
      ('u', -1.1662),
      ('z', -3.0412),
      ('?ĊĊ', -3.2912),
      ('?Ċ', -4.4787)]

### Process Results 

In [14]:
from src.process import process_results

metrics = process_results(results, data)

In [12]:
metrics

({'n_sample_loss_nexttoken': [4.6525,
   5.6084,
   4.816,
   5.1688,
   5.1201,
   5.1769,
   5.4363,
   5.0339,
   4.9996,
   4.9669,
   5.129,
   4.6426,
   5.0631,
   4.6479,
   4.5235,
   4.4589,
   4.4849,
   4.7921,
   4.5935,
   4.0769,
   4.3339,
   4.1614,
   3.7635,
   4.2636,
   4.1247,
   4.1475,
   3.9014,
   3.7303,
   4.1008,
   3.8901,
   4.0491,
   4.1169,
   3.3078,
   3.8488,
   3.2066,
   3.3555,
   3.5637,
   3.8082,
   3.64,
   3.4868,
   3.2251,
   3.4785,
   3.7426,
   3.1343,
   3.9341,
   3.4273,
   3.2557,
   2.8761,
   3.0596,
   3.8421,
   3.3378,
   3.3944,
   3.2745,
   2.9424,
   3.1114,
   3.2896,
   2.9795,
   2.946,
   3.3674,
   3.4704,
   3.0725,
   2.7411,
   3.3089,
   2.9771,
   3.2973,
   3.2183,
   2.8959,
   2.8472,
   2.8751,
   3.0143,
   3.4065,
   3.0049,
   3.0611,
   2.9456,
   3.1413,
   2.9311,
   3.322,
   3.2943,
   3.1124,
   2.7483,
   2.4876,
   2.5354,
   2.7447,
   3.06,
   2.8801,
   2.5432,
   3.1407,
   3.0808,
   3.0003,
  

### Visualize Results

In [ ]:
from src.viz import log_results

log_results(metrics=metrics, params=metadata)


In [30]:
models = [ "gpt-3.5-turbo", "gpt-4" ]
llama_models = ["meta-llama/Llama-2-7b-chat-hf", "meta-llama/Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-v0.1", "mistralai/Mistral-7B-Instruct-v0.1", "mistralai/Mistral-7B-Instruct-v0.2", "mistralai/Mistral-7B-v0.3", "mistralai/Ministral-8B-Instruct-2410"]

In [10]:
context = prompts[0]["context"]
queries = prompts[0]["query"]

context_message = ""
inputs = []
for i, query in enumerate(queries):
    context_message += context[i]
    prompt = model.gen_prompt(system, context_message, query)
    inputs.append(prompt)
# print("".join(inputs))
outputs, _ = model(inputs)
print(len(inputs), len(outputs))

for i, output in enumerate(outputs):
    prediction = output.outputs[0].text
    print(queries[i])
    print(f"{i}-----")
    print(prediction)
    print("======")

Processed prompts: 100%|██████████| 99/99 [00:00<00:00, 1358.53it/s, est. speed input: 650593.62 toks/s, output: 1361.68 toks/s]

99 99
X:S
Y:
0-----
V
X:S
Y:
1-----
V
X:X
Y:
2-----
Q
X:O
Y:
3-----
G
X:E
Y:
4-----
K
X:D
Y:
5-----
G
X:P
Y:
6-----
J
X:V
Y:
7-----
Z
X:Z
Y:
8-----
G
X:A
Y:
9-----
G
X:Q
Y:
10-----
G
X:C
Y:
11-----
G
X:P
Y:
12-----
J
X:H
Y:
13-----
G
X:L
Y:
14-----
G
X:E
Y:
15-----
Z
X:C
Y:
16-----
V
X:V
Y:
17-----
O
X:G
Y:
18-----
K
X:G
Y:
19-----
Z
X:R
Y:
20-----
K
X:D
Y:
21-----
W
X:F
Y:
22-----
G
X:Y
Y:
23-----
G
X:J
Y:
24-----
G
X:V
Y:
25-----
O
X:X
Y:
26-----
Q
X:Q
Y:
27-----
J
X:E
Y:
28-----
Z
X:R
Y:
29-----
K
X:Z
Y:
30-----
Z
X:D
Y:
31-----
W
X:F
Y:
32-----
Y
X:J
Y:
33-----
G
X:G
Y:
34-----
Z
X:E
Y:
35-----
Z
X:P
Y:
36-----
K
X:O
Y:
37-----
H
X:L
Y:
38-----
G
X:I
Y:
39-----
G
X:J
Y:
40-----
G
X:U
Y:
41-----
G
X:N
Y:
42-----
G
X:I
Y:
43-----
G
X:O
Y:
44-----
H
X:Q
Y:
45-----
J
X:C
Y:
46-----
V
X:Z
Y:
47-----
Z
X:Z
Y:
48-----
Z
X:N
Y:
49-----
G
X:G
Y:
50-----
Z
X:J
Y:
51-----
Z
X:N
Y:
52-----
G
X:Q
Y:
53-----
J
X:K
Y:
54-----
U
X:F
Y:
55-----
Y
X:C
Y:
56-----
V
X:Z
Y:
57-----
Z
X:D
Y:
58-----
W
X

In [ ]:
for prompt,  in prompts[0]["context"], prompts[0]["query"]):
    print(prompt)
    print("-"*40) 

# Predict next token in batch
sampling_params = SamplingParams(temperature=0.0, max_tokens=1, logprobs=6)
outputs = llm.generate(prompts, sampling_params)
res = []
# Print predictions
for i, output in enumerate(outputs):
    prediction = output.outputs[0].text.strip()
    out = output
    logprobs = [ (logprob.decoded_token, round(exp(logprob.logprob), 4)) for (token, logprob) in output.outputs[0].logprobs[0].items() ]
    res.append(logprobs[0][1])
    print(logprobs)
    print(f"Prompt {i+1}:\n{prompts[i]}\nPredicted y: {prediction} | Ground truth y: {Y[i+1]}\n{'-'*40}")
print(model_name, res)

In [ ]:

["meta-llama/Llama-3.1-8B-Instruct", [0.6571, 0.8955, 0.9482, 0.9892, 0.9654, 0.992, 0.9986, 0.9972, 0.9993, 0.9906, 0.9989, 0.9991, 0.9992, 0.9993, 0.9994, 0.9992, 0.9998, 0.9997, 0.9999, 0.9997, 0.9998, 0.9999, 0.9999, 0.9999, 0.9998, 0.9997, 0.9997, 0.9999, 0.9999]]
["mistralai/Mistral-7B-v0.1", [0.4459, 0.4893, 0.4478, 0.9, 0.9617, 0.9874, 0.9925, 0.9945, 0.9924, 0.987, 0.9917, 0.9946, 0.9956, 0.9961, 0.9923, 0.9939, 0.9974, 0.9983, 0.9977, 0.9933, 0.9961, 0.9984, 0.9989, 0.9984, 0.997, 0.9981, 0.9991, 0.9992, 0.9985]]
["mistralai/Mistral-7B-v0.3", [0.3512, 0.3977, 0.4113, 0.852, 0.9581, 0.9862, 0.9927, 0.9943, 0.9937, 0.9867, 0.9923, 0.9947, 0.9956, 0.9959, 0.9935, 0.9952, 0.9986, 0.9985, 0.9982, 0.9954, 0.9974, 0.9988, 0.999, 0.9986, 0.9981, 0.9985, 0.9991, 0.9992, 0.999]]
["mistralai/Mistral-7B-Instruct-v0.1", [0.558, 0.429, 0.7056, 0.941, 0.9819, 0.9969, 0.9979, 0.9968, 0.9803, 0.9958, 0.9976, 0.9979, 0.9981, 0.9876, 0.9982, 0.9985, 0.9988, 0.9984, 0.9968, 0.9982, 0.9996, 0.9997, 0.9997, 0.9983, 0.9996, 0.9996, 0.9997, 0.9997, 0.9986]]
["mistralai/Mistral-7B-Instruct-v0.2", [0.397, 0.5195, 0.922, 0.9728, 0.9746, 0.9915, 0.9981, 0.997, 0.9951, 0.9955, 0.9961, 0.9975, 0.999, 0.9981, 0.999, 0.9978, 0.9995, 0.9996, 0.9993, 0.9992, 0.9993, 0.9997, 0.9999, 0.9997, 0.9999, 0.9999, 0.9999, 0.9999, 0.9997]]
["mistralai/Ministral-8B-Instruct-2410", [0.3279, 0.4283, 0.9297, 0.9655, 0.9909, 0.9957, 0.998, 0.9986, 0.9981, 0.9945, 0.9961, 0.9981, 0.9983, 0.9989, 0.9988, 0.9989, 0.9996, 0.9997, 0.9995, 0.9981, 0.9987, 0.9996, 0.9997, 0.9995, 0.9991, 0.9996, 0.9999, 0.9999, 0.9998]]

['mistralai/Mistral-7B-v0.3',
 [0.3512,
  0.3977,
  0.4113,
  0.852,
  0.9581,
  0.9862,
  0.9927,
  0.9943,
  0.9937,
  0.9867,
  0.9923,
  0.9947,
  0.9956,
  0.9959,
  0.9935,
  0.9952,
  0.9986,
  0.9985,
  0.9982,
  0.9954,
  0.9974,
  0.9988,
  0.999,
  0.9986,
  0.9981,
  0.9985,
  0.9991,
  0.9992,
  0.999]]